# Study A reproducible 40-run analysis with missing-type-specific test rows

This standalone notebook consolidates the NoSpatial, XY, Z, and XYZ workflows for BM2 Study A. It generates mean and sample standard deviation over 40 paired seeds, 95% confidence intervals, constituent-model results, ensemble results, RMSE improvement tables, and paired Wilcoxon tests with Holm correction.

Important safeguards:

- The BM2 test set is not used for tuning or model selection.
- Test RMSE is evaluated on five predefined `rows` values for each missing type: MT1 uses `{0, 4, 8, 12, 16}`, MT2 uses `{1, 5, 9, 13, 17}`, MT3 uses `{2, 6, 10, 14, 18}`, and MT4 uses `{3, 7, 11, 15, 19}`.
- Preprocessing is fitted inside each cross-validation fold.
- Spatial features are assigned explicitly for every missing type.
- Duplicate training and validation IDs are removed before development data are combined.
- The pre-specified ensemble is the arithmetic mean of RF, GBR, and XGBoost predictions.


## Requirements and input files

Place the three CSV files in the same directory as this notebook, or enter their full paths in the settings cell. The test CSV must contain a unique integer `rows` column with values 0 through 19. Required packages are `numpy`, `pandas`, `scipy`, `scikit-learn`, `xgboost`, `joblib`, `matplotlib`, and `openpyxl`. The optional `psutil` package adds physical-core and memory information to the system table. If needed, install XGBoost in the active environment before starting the full experiment.


## Imports, constants, and experiment schema

In [1]:
from __future__ import annotations

import argparse
import json
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
import scipy
from scipy.stats import t as student_t
from scipy.stats import wilcoxon
import sklearn
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


CORE_FEATURES = [
    "Sr",
    "gamma_t",
    "e",
    "LL",
    "PL",
    "w",
    "PI",
    "LI",
    "e_squared",
    "w_by_LL",
    "Sr_log",
    "moisture_void_ratio",
]

SPATIAL_FEATURES = {
    "NoSpatial": [],
    "XY": ["x", "y"],
    "Z": ["z_depth"],
    "XYZ": ["x", "y", "z_depth"],
}

CONFIG_LABELS = {
    "NoSpatial": "Without Spatial",
    "XY": "With XY",
    "Z": "With Z",
    "XYZ": "With XYZ",
}

TARGET_ORDER = ["Cc", "Cv", "Pc", "E50", "su"]
CONFIG_ORDER = ["NoSpatial", "XY", "Z", "XYZ"]
MISSING_TYPE_ORDER = ["MT1", "MT2", "MT3", "MT4"]
TEST_ROWS_BY_MISSING_TYPE = {
    "MT1": [0, 4, 8, 12, 16],
    "MT2": [1, 5, 9, 13, 17],
    "MT3": [2, 6, 10, 14, 18],
    "MT4": [3, 7, 11, 15, 19],
}
MODEL_ORDER = ["RF", "GBR", "XGB", "Ensemble"]
SPLIT_ORDER = ["Development", "Test"]

MISSINGNESS_PATTERNS = {
    "MT1": {
        "known_mechanical": [],
        "targets": ["Cc", "Cv", "Pc", "E50", "su"],
        "polynomial_degree": 1,
    },
    "MT2": {
        "known_mechanical": ["su"],
        "targets": ["Cc", "Cv", "Pc", "E50"],
        "polynomial_degree": 2,
    },
    "MT3": {
        "known_mechanical": ["su", "E50"],
        "targets": ["Cc", "Cv", "Pc"],
        "polynomial_degree": 2,
    },
    "MT4": {
        "known_mechanical": ["Pc", "Cc", "Cv"],
        "targets": ["E50", "su"],
        "polynomial_degree": 2,
    },
}


@dataclass(frozen=True)
class ExperimentTask:
    configuration: str
    missing_type: str
    target: str
    features: tuple[str, ...]
    polynomial_degree: int


## Definitions and experiment schema

In [2]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate reproducible Study A mean +/- SD tables from 40 seeded runs."
    )
    parser.add_argument(
        "--train",
        default="BM2_Local_BID_11_minus_Table1_XYZ_merged.csv",
        help="Development/training CSV.",
    )
    parser.add_argument(
        "--validation",
        default="BM2_Local_val_BID_11_minus_Table1_XYZ_merged.csv",
        help="Validation CSV. Exact overlaps with training are deduplicated.",
    )
    parser.add_argument(
        "--test",
        default="BM2_Table_1_md_XYZ_merged_rows_rows.csv",
        help="The Study A BM2 test CSV with a unique integer rows column.",
    )
    parser.add_argument("--output-dir", default="StudyA_40run_results_Selected22Sept")
    parser.add_argument("--runs", type=int, default=40)
    parser.add_argument("--base-seed", type=int, default=2026)
    parser.add_argument("--tuning-seed", type=int, default=1729)
    parser.add_argument("--cv-folds", type=int, default=5)
    parser.add_argument(
        "--cv-strategy",
        choices=["group", "random"],
        default="group",
        help="Use borehole-grouped CV by default; choose random to mirror the notebooks.",
    )
    parser.add_argument(
        "--search-profile",
        choices=["expanded", "manuscript", "smoke"],
        default="expanded",
        help="Expanded addresses reviewer concerns; manuscript mirrors Table 1; smoke is for testing.",
    )
    parser.add_argument(
        "--search-iterations",
        type=int,
        default=20,
        help="Randomized-search iterations per model for the expanded profile.",
    )
    parser.add_argument(
        "--tuning-score",
        choices=["rmse", "r2"],
        default="rmse",
        help="Hyperparameter selection objective. RMSE is aligned with the manuscript tables.",
    )
    parser.add_argument("--n-jobs", type=int, default=-1)
    parser.add_argument("--huber-delta", type=float, default=1.0)
    parser.add_argument(
        "--configurations",
        nargs="+",
        choices=CONFIG_ORDER,
        default=CONFIG_ORDER,
    )
    parser.add_argument(
        "--missing-types",
        nargs="+",
        choices=MISSING_TYPE_ORDER,
        default=MISSING_TYPE_ORDER,
    )
    parser.add_argument(
        "--targets",
        nargs="+",
        choices=TARGET_ORDER,
        default=None,
        help="Optional target subset for diagnostics or smoke runs.",
    )
    parser.add_argument(
        "--models",
        nargs="+",
        choices=["RF", "GBR", "XGB"],
        default=["RF", "GBR", "XGB"],
    )
    parser.add_argument(
        "--overlap-policy",
        choices=["drop-train", "error", "allow"],
        default="drop-train",
        help="How to handle exact global_id overlap between training and validation.",
    )
    parser.add_argument(
        "--save-models",
        action="store_true",
        help="Save every fitted constituent model. This can require substantial disk space.",
    )
    parser.add_argument(
        "--validate-only",
        action="store_true",
        help="Validate inputs and write manifests without fitting models.",
    )
    return parser.parse_args()


## Arguments, input loading, and leakage checks

In [3]:
def resolve_input_path(raw_path: str) -> Path:
    supplied = Path(raw_path).expanduser()
    script_location = globals().get("__file__")
    script_dir = Path(script_location).resolve().parent if script_location else Path.cwd()
    candidates = [supplied]
    if not supplied.is_absolute():
        candidates.extend(
            [
                Path.cwd() / supplied,
                script_dir / supplied,
                script_dir / "upload" / supplied.name,
            ]
        )
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {raw_path!r}. Supply an explicit path with --train, --validation, or --test."
    )


def load_and_engineer(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed:")].copy()
    required_raw = {
        "global_id",
        "borehole_id",
        "Sr",
        "gamma_t",
        "e",
        "LL",
        "PL",
        "w",
        "su",
        "E50",
        "Pc",
        "Cc",
        "Cv",
        "x",
        "y",
        "z_depth",
    }
    missing = sorted(required_raw.difference(frame.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    frame["PI"] = frame["LL"] - frame["PL"]
    frame["LI"] = (frame["w"] - frame["PL"]) / (frame["PI"] + 1e-5)
    frame["e_squared"] = frame["e"] ** 2
    frame["w_by_LL"] = frame["w"] / (frame["LL"] + 1e-5)
    if (frame["Sr"] + 1e-5 <= 0).any():
        raise ValueError(f"{path.name} contains Sr values that cannot be log transformed.")
    frame["Sr_log"] = np.log(frame["Sr"] + 1e-5)
    frame["moisture_void_ratio"] = frame["w"] * frame["e"]
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)

    check_columns = sorted(
        set(CORE_FEATURES)
        | {"x", "y", "z_depth", "su", "E50", "Pc", "Cc", "Cv"}
    )
    missing_counts = frame[check_columns].isna().sum()
    missing_counts = missing_counts[missing_counts > 0]
    if not missing_counts.empty:
        raise ValueError(
            f"{path.name} has missing or infinite modeling values: {missing_counts.to_dict()}"
        )
    return frame


def validate_test_row_mapping(
    test: pd.DataFrame,
) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    if "rows" not in test.columns:
        raise ValueError(
            "The test CSV must contain a 'rows' column for missing-type-specific evaluation."
        )

    numeric_rows = pd.to_numeric(test["rows"], errors="raise")
    if not np.allclose(numeric_rows.to_numpy(float), np.round(numeric_rows.to_numpy(float))):
        raise ValueError("The test CSV 'rows' values must be integers.")

    validated = test.copy()
    validated["rows"] = numeric_rows.astype("int64")
    duplicated = validated.loc[validated["rows"].duplicated(keep=False), "rows"].tolist()
    if duplicated:
        raise ValueError(f"The test CSV contains duplicated rows values: {sorted(set(duplicated))}")

    available = set(validated["rows"].tolist())
    required = set().union(*[set(values) for values in TEST_ROWS_BY_MISSING_TYPE.values()])
    missing = sorted(required.difference(available))
    if missing:
        raise ValueError(f"The test CSV is missing required rows values: {missing}")

    audit = [
        {"item": "test_rows_column_unique_values", "value": validated["rows"].nunique()},
        {"item": "test_required_rows_available", "value": len(required)},
    ]
    for missing_type in MISSING_TYPE_ORDER:
        selected_rows = TEST_ROWS_BY_MISSING_TYPE[missing_type]
        audit.extend(
            [
                {
                    "item": f"{missing_type}_test_rows_selected",
                    "value": ", ".join(str(value) for value in selected_rows),
                },
                {
                    "item": f"{missing_type}_test_record_count",
                    "value": len(selected_rows),
                },
            ]
        )
    return validated, audit


def select_test_rows(test: pd.DataFrame, missing_type: str) -> pd.DataFrame:
    if missing_type not in TEST_ROWS_BY_MISSING_TYPE:
        raise KeyError(f"No test-row mapping is defined for {missing_type!r}")
    requested_rows = TEST_ROWS_BY_MISSING_TYPE[missing_type]
    selected = (
        test.set_index("rows", drop=False)
        .loc[requested_rows]
        .reset_index(drop=True)
        .copy()
    )
    if len(selected) != len(requested_rows):
        raise RuntimeError(
            f"Expected {len(requested_rows)} test records for {missing_type}, found {len(selected)}"
        )
    return selected


def prepare_splits(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    overlap_policy: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[dict[str, Any]]]:
    train_ids = set(train["global_id"])
    validation_ids = set(validation["global_id"])
    test_ids = set(test["global_id"])
    train_validation_overlap = train_ids & validation_ids
    development_test_overlap = (train_ids | validation_ids) & test_ids

    if development_test_overlap:
        raise ValueError(
            "Test leakage detected: global_id values occur in both development and test data: "
            f"{sorted(development_test_overlap)[:20]}"
        )
    if train_validation_overlap and overlap_policy == "error":
        raise ValueError(
            f"Training and validation overlap by {len(train_validation_overlap)} global_id values. "
            "Use --overlap-policy drop-train to deduplicate safely."
        )
    if train_validation_overlap and overlap_policy == "drop-train":
        train = train.loc[~train["global_id"].isin(train_validation_overlap)].copy()
    elif train_validation_overlap and overlap_policy == "allow":
        warnings.warn(
            f"Allowing {len(train_validation_overlap)} overlapping train/validation IDs. "
            "They will still be deduplicated when the combined development set is created.",
            RuntimeWarning,
        )

    development = pd.concat(
        [train.assign(source_split="train"), validation.assign(source_split="validation")],
        ignore_index=True,
    )
    development = development.drop_duplicates(subset="global_id", keep="last").reset_index(drop=True)
    test = test.reset_index(drop=True).copy()
    test["source_split"] = "test"

    audit = [
        {"item": "train_rows_supplied", "value": len(train) + (len(train_validation_overlap) if overlap_policy == "drop-train" else 0)},
        {"item": "validation_rows_supplied", "value": len(validation)},
        {"item": "test_rows_supplied", "value": len(test)},
        {"item": "train_validation_overlapping_ids", "value": len(train_validation_overlap)},
        {"item": "development_test_overlapping_ids", "value": len(development_test_overlap)},
        {"item": "development_unique_rows_used", "value": len(development)},
        {"item": "test_unique_rows_used", "value": test["global_id"].nunique()},
        {"item": "development_boreholes", "value": development["borehole_id"].nunique()},
        {"item": "test_boreholes", "value": test["borehole_id"].nunique()},
    ]
    return train, development, test, audit


def iter_tasks(args: argparse.Namespace) -> Iterable[ExperimentTask]:
    requested_targets = set(args.targets) if args.targets else None
    for missing_type in MISSING_TYPE_ORDER:
        if missing_type not in args.missing_types:
            continue
        pattern = MISSINGNESS_PATTERNS[missing_type]
        for configuration in CONFIG_ORDER:
            if configuration not in args.configurations:
                continue
            features = tuple(
                CORE_FEATURES
                + SPATIAL_FEATURES[configuration]
                + list(pattern["known_mechanical"])
            )
            for target in pattern["targets"]:
                if requested_targets is not None and target not in requested_targets:
                    continue
                yield ExperimentTask(
                    configuration=configuration,
                    missing_type=missing_type,
                    target=target,
                    features=features,
                    polynomial_degree=int(pattern["polynomial_degree"]),
                )


## Model construction and hyperparameter search

In [4]:
def get_xgb_regressor(seed: int) -> Any:
    try:
        import xgboost
        from xgboost import XGBRegressor
    except ImportError as exc:
        raise ImportError(
            "XGBoost is required for the three-model manuscript ensemble. "
            "Install it with `pip install xgboost`, or use --models RF GBR only for a smoke test."
        ) from exc
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )


def base_model(model_name: str, seed: int) -> Any:
    if model_name == "RF":
        return RandomForestRegressor(random_state=seed, n_jobs=1)
    if model_name == "GBR":
        return GradientBoostingRegressor(random_state=seed, loss="squared_error")
    if model_name == "XGB":
        return get_xgb_regressor(seed)
    raise KeyError(model_name)


def search_space(model_name: str, profile: str) -> dict[str, list[Any]]:
    prefix = "regressor__model__"
    if profile == "smoke":
        spaces = {
            "RF": {"n_estimators": [20], "max_depth": [5], "min_samples_split": [2]},
            "GBR": {"n_estimators": [20], "learning_rate": [0.05], "max_depth": [3]},
            "XGB": {"n_estimators": [20], "learning_rate": [0.05], "max_depth": [3]},
        }
    elif profile == "manuscript":
        spaces = {
            "RF": {
                "n_estimators": [100, 200, 500],
                "max_depth": [5, 10],
                "min_samples_split": [2, 5],
            },
            "GBR": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.05, 0.1],
                "max_depth": [3, 5],
            },
            "XGB": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.05, 0.1],
                "max_depth": [3, 5],
            },
        }
    else:
        spaces = {
            "RF": {
                "n_estimators": [200, 500, 800],
                "max_depth": [None, 5, 10, 20],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "max_features": [1.0, 0.5, "sqrt"],
            },
            "GBR": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.03, 0.05, 0.1],
                "max_depth": [2, 3, 5],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "subsample": [0.8, 1.0],
                "max_features": [None, "sqrt"],
            },
            "XGB": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.03, 0.05, 0.1],
                "max_depth": [3, 5, 8],
                "min_child_weight": [1, 3, 5],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
                "reg_alpha": [0.0, 0.1],
                "reg_lambda": [1.0, 5.0],
            },
        }
    return {prefix + key: value for key, value in spaces[model_name].items()}


def build_estimator(model_name: str, degree: int, seed: int) -> TransformedTargetRegressor:
    pipeline = Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scale", StandardScaler()),
            ("model", base_model(model_name, seed)),
        ]
    )
    return TransformedTargetRegressor(regressor=pipeline, transformer=StandardScaler())


def make_cv(
    strategy: str,
    folds: int,
    seed: int,
    groups: pd.Series,
) -> tuple[Any, np.ndarray | None]:
    if strategy == "random":
        return KFold(n_splits=folds, shuffle=True, random_state=seed), None
    unique_groups = groups.nunique()
    if unique_groups < folds:
        raise ValueError(f"Grouped CV requested {folds} folds but only {unique_groups} groups exist.")
    try:
        splitter = GroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    except TypeError:
        splitter = GroupKFold(n_splits=folds)
    return splitter, groups.to_numpy()


def tune_task_models(
    task: ExperimentTask,
    development: pd.DataFrame,
    args: argparse.Namespace,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    X = development.loc[:, task.features]
    y = development.loc[:, task.target]
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development["borehole_id"],
    )
    scoring = "neg_root_mean_squared_error" if args.tuning_score == "rmse" else "r2"
    estimators: dict[str, Any] = {}
    records: list[dict[str, Any]] = []

    for model_name in args.models:
        estimator = build_estimator(model_name, task.polynomial_degree, args.tuning_seed)
        params = search_space(model_name, args.search_profile)
        if args.search_profile == "expanded":
            search = RandomizedSearchCV(
                estimator=estimator,
                param_distributions=params,
                n_iter=args.search_iterations,
                scoring=scoring,
                cv=cv,
                random_state=args.tuning_seed,
                n_jobs=args.n_jobs,
                refit=True,
                return_train_score=False,
            )
        else:
            search = GridSearchCV(
                estimator=estimator,
                param_grid=params,
                scoring=scoring,
                cv=cv,
                n_jobs=args.n_jobs,
                refit=True,
                return_train_score=False,
            )
        fit_kwargs = {"groups": groups} if groups is not None else {}
        started = time.perf_counter()
        search.fit(X, y, **fit_kwargs)
        elapsed = time.perf_counter() - started
        estimators[model_name] = clone(search.best_estimator_)
        records.append(
            {
                "configuration": task.configuration,
                "configuration_label": CONFIG_LABELS[task.configuration],
                "missing_type": task.missing_type,
                "target": task.target,
                "model": model_name,
                "features": ", ".join(task.features),
                "raw_feature_count": len(task.features),
                "polynomial_degree": task.polynomial_degree,
                "expanded_feature_count": _expanded_feature_count(
                    len(task.features), task.polynomial_degree
                ),
                "cv_strategy": args.cv_strategy,
                "cv_folds": args.cv_folds,
                "tuning_score": args.tuning_score,
                "search_profile": args.search_profile,
                "best_cv_score": float(search.best_score_),
                "best_parameters": json.dumps(search.best_params_, sort_keys=True),
                "tuning_seconds": elapsed,
            }
        )
    return estimators, records


def _expanded_feature_count(raw_count: int, degree: int) -> int:
    if degree == 1:
        return raw_count
    if degree == 2:
        return raw_count + raw_count * (raw_count + 1) // 2
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    poly.fit(np.zeros((1, raw_count)))
    return int(poly.n_output_features_)


def reseed_estimator(estimator: Any, seed: int, n_jobs: int) -> Any:
    fitted = clone(estimator)
    available = fitted.get_params(deep=True)
    updates: dict[str, Any] = {}
    for name in available:
        if name.endswith("random_state"):
            updates[name] = seed
        elif name.endswith("n_jobs"):
            updates[name] = n_jobs
    if updates:
        fitted.set_params(**updates)
    return fitted


## Repeated-run evaluation and metrics

Development metrics use the complete development set. Test predictions and test metrics use only the five mapped `rows` values for the task's missing type.


In [5]:
def robust_l2(y_true: np.ndarray, y_pred: np.ndarray, delta: float) -> float:
    residual = np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)
    absolute = np.abs(residual)
    loss = np.where(
        absolute <= delta,
        0.5 * residual**2,
        delta * (absolute - 0.5 * delta),
    )
    return float(np.mean(loss))


def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    huber_delta: float,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    l2 = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(l2))
    denominator = float(np.mean(np.abs(y_true)))
    rmse_percent = np.nan if denominator == 0 else 100.0 * rmse / denominator
    return {
        "rmse": rmse,
        "rmse_percent": rmse_percent,
        "l2": l2,
        "robust_l2": robust_l2(y_true, y_pred, huber_delta),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def evaluate_task(
    task: ExperimentTask,
    tuned_estimators: dict[str, Any],
    development: pd.DataFrame,
    test: pd.DataFrame,
    args: argparse.Namespace,
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    task_test = select_test_rows(test, task.missing_type)
    X_development = development.loc[:, task.features]
    y_development = development.loc[:, task.target].to_numpy()
    X_test = task_test.loc[:, task.features]
    y_test = task_test.loc[:, task.target].to_numpy()
    metric_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []

    for run_index in range(args.runs):
        seed = args.base_seed + run_index
        predictions: dict[str, dict[str, np.ndarray]] = {
            "Development": {},
            "Test": {},
        }
        for model_name, template in tuned_estimators.items():
            estimator = reseed_estimator(template, seed, args.n_jobs)
            started = time.perf_counter()
            estimator.fit(X_development, y_development)
            fit_seconds = time.perf_counter() - started
            predictions["Development"][model_name] = estimator.predict(X_development)
            predictions["Test"][model_name] = estimator.predict(X_test)
            if args.save_models:
                model_dir = output_dir / "models" / task.configuration / task.missing_type / task.target
                model_dir.mkdir(parents=True, exist_ok=True)
                joblib.dump(estimator, model_dir / f"{model_name}_seed_{seed}.joblib")

            for split, truth in [("Development", y_development), ("Test", y_test)]:
                values = calculate_metrics(
                    truth,
                    predictions[split][model_name],
                    args.huber_delta,
                )
                metric_rows.append(
                    _metric_record(task, run_index, seed, model_name, split, len(truth), fit_seconds, values)
                )

        for split, truth in [("Development", y_development), ("Test", y_test)]:
            ensemble_prediction = np.mean(
                np.column_stack([predictions[split][name] for name in args.models]),
                axis=1,
            )
            predictions[split]["Ensemble"] = ensemble_prediction
            values = calculate_metrics(truth, ensemble_prediction, args.huber_delta)
            metric_rows.append(
                _metric_record(task, run_index, seed, "Ensemble", split, len(truth), np.nan, values)
            )

        for row_position, (_, source_row) in enumerate(task_test.iterrows()):
            record = {
                "run": run_index + 1,
                "seed": seed,
                "configuration": task.configuration,
                "configuration_label": CONFIG_LABELS[task.configuration],
                "missing_type": task.missing_type,
                "target": task.target,
                "test_row": int(source_row["rows"]),
                "global_id": source_row["global_id"],
                "borehole_id": source_row["borehole_id"],
                "actual": y_test[row_position],
            }
            for model_name in args.models + ["Ensemble"]:
                record[f"prediction_{model_name}"] = predictions["Test"][model_name][row_position]
            prediction_rows.append(record)

        print(
            f"  completed {task.configuration} {task.missing_type} {task.target}, "
            f"run {run_index + 1}/{args.runs}, seed={seed}",
            flush=True,
        )
    return metric_rows, prediction_rows


def _metric_record(
    task: ExperimentTask,
    run_index: int,
    seed: int,
    model: str,
    split: str,
    sample_count: int,
    fit_seconds: float,
    values: dict[str, float],
) -> dict[str, Any]:
    return {
        "run": run_index + 1,
        "seed": seed,
        "configuration": task.configuration,
        "configuration_label": CONFIG_LABELS[task.configuration],
        "missing_type": task.missing_type,
        "target": task.target,
        "model": model,
        "split": split,
        "n": sample_count,
        "fit_seconds": fit_seconds,
        **values,
    }


## Uncertainty summaries and paired statistical tests

In [6]:
def summarize_metrics(run_metrics: pd.DataFrame) -> pd.DataFrame:
    identifiers = [
        "configuration",
        "configuration_label",
        "missing_type",
        "target",
        "model",
        "split",
        "n",
    ]
    metrics = ["rmse", "rmse_percent", "l2", "robust_l2", "mae", "r2"]
    grouped = run_metrics.groupby(identifiers, sort=False, dropna=False)
    rows: list[dict[str, Any]] = []
    for keys, group in grouped:
        row = dict(zip(identifiers, keys))
        row["runs"] = len(group)
        critical = student_t.ppf(0.975, df=max(len(group) - 1, 1))
        for metric in metrics:
            values = group[metric].to_numpy(dtype=float)
            mean = float(np.mean(values))
            std = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
            margin = critical * std / np.sqrt(len(values)) if len(values) > 1 else np.nan
            row[f"{metric}_mean"] = mean
            row[f"{metric}_sd"] = std
            row[f"{metric}_ci95_low"] = mean - margin if len(values) > 1 else np.nan
            row[f"{metric}_ci95_high"] = mean + margin if len(values) > 1 else np.nan
        rows.append(row)
    result = pd.DataFrame(rows)
    return sort_results(result)


def calculate_improvements(run_metrics: pd.DataFrame) -> pd.DataFrame:
    ensemble = run_metrics.loc[run_metrics["model"] == "Ensemble"].copy()
    baseline = ensemble.loc[ensemble["configuration"] == "NoSpatial", [
        "run", "seed", "missing_type", "target", "split", "rmse", "rmse_percent"
    ]].rename(columns={"rmse": "baseline_rmse", "rmse_percent": "baseline_rmse_percent"})
    spatial = ensemble.loc[ensemble["configuration"] != "NoSpatial"].copy()
    merged = spatial.merge(
        baseline,
        on=["run", "seed", "missing_type", "target", "split"],
        how="left",
        validate="many_to_one",
    )
    if merged[["baseline_rmse", "baseline_rmse_percent"]].isna().any().any():
        raise ValueError("Improvement tables require NoSpatial results for every selected task.")
    merged["rmse_improvement_percent"] = 100.0 * (
        merged["baseline_rmse"] - merged["rmse"]
    ) / merged["baseline_rmse"]
    merged["rmse_percent_improvement_percent"] = 100.0 * (
        merged["baseline_rmse_percent"] - merged["rmse_percent"]
    ) / merged["baseline_rmse_percent"]
    columns = [
        "run",
        "seed",
        "configuration",
        "configuration_label",
        "missing_type",
        "target",
        "split",
        "baseline_rmse",
        "rmse",
        "rmse_improvement_percent",
        "baseline_rmse_percent",
        "rmse_percent",
        "rmse_percent_improvement_percent",
    ]
    return sort_results(merged[columns])


def summarize_improvements(improvements: pd.DataFrame) -> pd.DataFrame:
    identifiers = ["configuration", "configuration_label", "missing_type", "target", "split"]
    rows: list[dict[str, Any]] = []
    for keys, group in improvements.groupby(identifiers, sort=False):
        row = dict(zip(identifiers, keys))
        row["runs"] = len(group)
        for metric in ["rmse_improvement_percent", "rmse_percent_improvement_percent"]:
            values = group[metric].to_numpy(float)
            row[f"{metric}_mean"] = float(values.mean())
            row[f"{metric}_sd"] = float(values.std(ddof=1)) if len(values) > 1 else np.nan
        rows.append(row)
    return sort_results(pd.DataFrame(rows))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty_like(p_values)
    running = 0.0
    total = len(p_values)
    for rank, original_index in enumerate(order):
        candidate = min(1.0, (total - rank) * p_values[original_index])
        running = max(running, candidate)
        adjusted_sorted[original_index] = running
    return adjusted_sorted


def paired_tests(run_metrics: pd.DataFrame) -> pd.DataFrame:
    ensemble_test = run_metrics.loc[
        (run_metrics["model"] == "Ensemble") & (run_metrics["split"] == "Test")
    ].copy()
    rows: list[dict[str, Any]] = []
    for (missing_type, target, configuration), spatial in ensemble_test.loc[
        ensemble_test["configuration"] != "NoSpatial"
    ].groupby(["missing_type", "target", "configuration"], sort=False):
        baseline = ensemble_test.loc[
            (ensemble_test["missing_type"] == missing_type)
            & (ensemble_test["target"] == target)
            & (ensemble_test["configuration"] == "NoSpatial"),
            ["seed", "rmse"],
        ].rename(columns={"rmse": "baseline_rmse"})
        paired = spatial[["seed", "rmse"]].merge(baseline, on="seed", validate="one_to_one")
        difference = paired["rmse"].to_numpy() - paired["baseline_rmse"].to_numpy()
        if len(difference) < 2 or np.allclose(difference, 0):
            statistic, p_value = np.nan, 1.0
        else:
            result = wilcoxon(difference, alternative="two-sided", zero_method="wilcox")
            statistic, p_value = float(result.statistic), float(result.pvalue)
        sd = float(np.std(difference, ddof=1)) if len(difference) > 1 else np.nan
        rows.append(
            {
                "missing_type": missing_type,
                "target": target,
                "configuration": configuration,
                "configuration_label": CONFIG_LABELS[configuration],
                "paired_runs": len(difference),
                "mean_rmse_difference_spatial_minus_baseline": float(np.mean(difference)),
                "median_rmse_difference_spatial_minus_baseline": float(np.median(difference)),
                "paired_effect_size_dz": float(np.mean(difference) / sd) if sd and np.isfinite(sd) else np.nan,
                "wilcoxon_statistic": statistic,
                "p_value_two_sided": p_value,
            }
        )
    result = pd.DataFrame(rows)
    if not result.empty:
        result["p_value_holm"] = holm_adjust(result["p_value_two_sided"].to_numpy())
        result["significant_holm_0.05"] = result["p_value_holm"] < 0.05
    return sort_results(result)


## Manuscript tables and exports

In [7]:
def mean_sd_text(mean: float, sd: float, target: str, metric: str) -> str:
    if pd.isna(mean):
        return ""
    if target == "Cc" and metric not in {"rmse_percent", "r2"}:
        digits = 4
    elif metric in {"rmse_percent", "r2"}:
        digits = 3
    elif target in {"E50", "Cv"}:
        digits = 2
    else:
        digits = 3
    if pd.isna(sd):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} +/- {sd:.{digits}f}"


def manuscript_performance_table(summary: pd.DataFrame, missing_type: str) -> pd.DataFrame:
    data = summary.loc[
        (summary["model"] == "Ensemble") & (summary["missing_type"] == missing_type)
    ].copy()
    rows: list[dict[str, Any]] = []
    metrics = ["rmse", "rmse_percent", "l2", "robust_l2"]
    for configuration in CONFIG_ORDER:
        for target in TARGET_ORDER:
            subset = data.loc[
                (data["configuration"] == configuration) & (data["target"] == target)
            ]
            if subset.empty:
                continue
            row: dict[str, Any] = {
                "Configuration": CONFIG_LABELS[configuration],
                "Target": target,
            }
            for split in SPLIT_ORDER:
                split_row = subset.loc[subset["split"] == split]
                if split_row.empty:
                    continue
                values = split_row.iloc[0]
                for metric in metrics:
                    label = {
                        "rmse": "RMSE",
                        "rmse_percent": "RMSE%",
                        "l2": "L2",
                        "robust_l2": "Robust L2",
                    }[metric]
                    row[f"{split} {label}"] = mean_sd_text(
                        values[f"{metric}_mean"], values[f"{metric}_sd"], target, metric
                    )
            rows.append(row)
    return pd.DataFrame(rows)


def manuscript_ablation_table(summary: pd.DataFrame) -> pd.DataFrame:
    data = summary.loc[summary["split"] == "Test"].copy()
    rows: list[dict[str, Any]] = []
    for missing_type in MISSING_TYPE_ORDER:
        for target in TARGET_ORDER:
            for configuration in CONFIG_ORDER:
                subset = data.loc[
                    (data["missing_type"] == missing_type)
                    & (data["target"] == target)
                    & (data["configuration"] == configuration)
                ]
                if subset.empty:
                    continue
                row = {
                    "Missing Type": missing_type,
                    "Target": target,
                    "Configuration": CONFIG_LABELS[configuration],
                }
                for model in MODEL_ORDER:
                    model_row = subset.loc[subset["model"] == model]
                    if model_row.empty:
                        continue
                    values = model_row.iloc[0]
                    row[f"{model} test RMSE"] = mean_sd_text(
                        values["rmse_mean"], values["rmse_sd"], target, "rmse"
                    )
                rows.append(row)
    return pd.DataFrame(rows)


def manuscript_improvement_table(improvement_summary: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for missing_type in MISSING_TYPE_ORDER:
        for target in TARGET_ORDER:
            for configuration in ["XY", "Z", "XYZ"]:
                subset = improvement_summary.loc[
                    (improvement_summary["missing_type"] == missing_type)
                    & (improvement_summary["target"] == target)
                    & (improvement_summary["configuration"] == configuration)
                ]
                if subset.empty:
                    continue
                row = {
                    "Missing Type": missing_type,
                    "Target": target,
                    "Configuration": CONFIG_LABELS[configuration],
                }
                for split in SPLIT_ORDER:
                    split_row = subset.loc[subset["split"] == split]
                    if split_row.empty:
                        continue
                    values = split_row.iloc[0]
                    mean = values["rmse_improvement_percent_mean"]
                    sd = values["rmse_improvement_percent_sd"]
                    row[f"{split} RMSE improvement (%)"] = (
                        f"{mean:.2f} +/- {sd:.2f}" if pd.notna(sd) else f"{mean:.2f}"
                    )
                rows.append(row)
    return pd.DataFrame(rows)


def sort_results(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    result = frame.copy()
    sort_columns: list[str] = []
    temporary: list[str] = []
    mappings = {
        "configuration": CONFIG_ORDER,
        "missing_type": MISSING_TYPE_ORDER,
        "target": TARGET_ORDER,
        "model": MODEL_ORDER,
        "split": SPLIT_ORDER,
    }
    for column, order in mappings.items():
        if column in result.columns:
            helper = f"__order_{column}"
            result[helper] = pd.Categorical(result[column], categories=order, ordered=True)
            sort_columns.append(helper)
            temporary.append(helper)
    for column in ["run", "seed", "test_row"]:
        if column in result.columns:
            sort_columns.append(column)
    if sort_columns:
        result = result.sort_values(sort_columns, kind="stable")
    return result.drop(columns=temporary).reset_index(drop=True)


def write_excel(output_path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    try:
        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            for sheet_name, frame in sheets.items():
                safe_name = sheet_name[:31]
                frame.to_excel(writer, sheet_name=safe_name, index=False)
                worksheet = writer.book[safe_name]
                worksheet.freeze_panes = "A2"
                worksheet.auto_filter.ref = worksheet.dimensions
                for column_cells in worksheet.columns:
                    width = min(
                        55,
                        max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
                        + 2,
                    )
                    worksheet.column_dimensions[column_cells[0].column_letter].width = width
    except ImportError as exc:
        warnings.warn(f"Excel workbook was not written because openpyxl is unavailable: {exc}")


def software_versions() -> dict[str, str]:
    versions = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__,
    }
    try:
        import xgboost

        versions["xgboost"] = xgboost.__version__
    except ImportError:
        versions["xgboost"] = "not installed"
    return versions


def write_manifests(
    output_dir: Path,
    args: argparse.Namespace,
    tasks: list[ExperimentTask],
    audit: list[dict[str, Any]],
    input_paths: dict[str, Path],
) -> None:
    pd.DataFrame(audit).to_csv(output_dir / "sample_count_and_overlap_audit.csv", index=False)
    feature_rows = []
    for task in tasks:
        feature_rows.append(
            {
                **asdict(task),
                "configuration_label": CONFIG_LABELS[task.configuration],
                "features": ", ".join(task.features),
                "raw_feature_count": len(task.features),
                "expanded_feature_count": _expanded_feature_count(
                    len(task.features), task.polynomial_degree
                ),
            }
        )
    pd.DataFrame(feature_rows).to_csv(output_dir / "feature_manifest.csv", index=False)
    manifest = {
        "arguments": vars(args),
        "input_paths": {key: str(value) for key, value in input_paths.items()},
        "software_versions": software_versions(),
        "metric_definitions": {
            "rmse": "sqrt(mean((actual - predicted)^2))",
            "rmse_percent": "100 * RMSE / mean(abs(actual))",
            "l2": "mean((actual - predicted)^2)",
            "robust_l2": (
                "mean Huber loss: 0.5*r^2 when abs(r)<=delta; "
                "delta*(abs(r)-0.5*delta) otherwise"
            ),
            "huber_delta": args.huber_delta,
            "standard_deviation": "sample SD across paired random seeds (ddof=1)",
        },
        "selection_rule": (
            "Hyperparameters selected using development data only. Test labels are used only "
            "for locked final evaluation. The ensemble is a pre-specified arithmetic mean."
        ),
    }
    (output_dir / "experiment_manifest.json").write_text(
        json.dumps(manifest, indent=2, default=str), encoding="utf-8"
    )


## Experiment driver

In [8]:
def main() -> int:
    pipeline_started = time.perf_counter()
    args = parse_args()
    if args.runs < 2 and not args.validate_only:
        warnings.warn("At least two runs are required to estimate a standard deviation.")
    if args.huber_delta <= 0:
        raise ValueError("--huber-delta must be positive.")
    if args.search_iterations < 1:
        raise ValueError("--search-iterations must be at least 1.")

    paths = {
        "train": resolve_input_path(args.train),
        "validation": resolve_input_path(args.validation),
        "test": resolve_input_path(args.test),
    }
    output_dir = Path(args.output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Loading and validating Study A data...", flush=True)
    supplied_train = load_and_engineer(paths["train"])
    validation = load_and_engineer(paths["validation"])
    test = load_and_engineer(paths["test"])
    test, test_row_audit = validate_test_row_mapping(test)
    _, development, test, audit = prepare_splits(
        supplied_train, validation, test, args.overlap_policy
    )
    audit.extend(test_row_audit)
    tasks = list(iter_tasks(args))
    if not tasks:
        raise ValueError("No experiment tasks remain after applying the requested filters.")
    write_manifests(output_dir, args, tasks, audit, paths)

    print(pd.DataFrame(audit).to_string(index=False), flush=True)
    print(f"Experiment tasks: {len(tasks)}", flush=True)
    if args.validate_only:
        pipeline_seconds = time.perf_counter() - pipeline_started
        pd.DataFrame(
            [{
                "stage": "Input validation and manifest generation",
                "elapsed_seconds": pipeline_seconds,
                "elapsed_minutes": pipeline_seconds / 60.0,
            }]
        ).to_csv(output_dir / "pipeline_wall_clock_time.csv", index=False)
        print(f"Validation complete. Manifests written to {output_dir}")
        return 0
    if "XGB" in args.models:
        # Fail before any expensive tuning if the required package is absent.
        get_xgb_regressor(args.tuning_seed)
    if "NoSpatial" not in args.configurations:
        warnings.warn("NoSpatial was omitted, so improvement and paired-test tables cannot be generated.")

    all_metrics: list[dict[str, Any]] = []
    all_predictions: list[dict[str, Any]] = []
    all_hyperparameters: list[dict[str, Any]] = []

    for task_number, task in enumerate(tasks, start=1):
        print(
            f"[{task_number}/{len(tasks)}] Tuning {task.configuration} "
            f"{task.missing_type} {task.target} using development data only...",
            flush=True,
        )
        tuned, tuning_records = tune_task_models(task, development, args)
        all_hyperparameters.extend(tuning_records)
        metric_rows, prediction_rows = evaluate_task(
            task, tuned, development, test, args, output_dir
        )
        all_metrics.extend(metric_rows)
        all_predictions.extend(prediction_rows)

        pd.DataFrame(all_hyperparameters).to_csv(
            output_dir / "selected_hyperparameters_checkpoint.csv", index=False
        )
        pd.DataFrame(all_metrics).to_csv(
            output_dir / "run_metrics_checkpoint.csv", index=False
        )

    run_metrics = sort_results(pd.DataFrame(all_metrics))
    predictions = sort_results(pd.DataFrame(all_predictions))
    hyperparameters = sort_results(pd.DataFrame(all_hyperparameters))
    summary = summarize_metrics(run_metrics)

    run_metrics.to_csv(output_dir / "run_metrics_all_models.csv", index=False)
    predictions.to_csv(output_dir / "test_predictions_all_runs.csv", index=False)
    hyperparameters.to_csv(output_dir / "selected_hyperparameters.csv", index=False)
    summary.to_csv(output_dir / "summary_metrics_mean_sd_ci95.csv", index=False)

    table2 = manuscript_performance_table(summary, "MT1")
    table3 = manuscript_ablation_table(summary)
    table2.to_csv(output_dir / "Table2_StudyA_MT1_mean_SD.csv", index=False)
    table3.to_csv(output_dir / "Table3_StudyA_model_ablation_mean_SD.csv", index=False)

    per_missing_type: dict[str, pd.DataFrame] = {}
    for missing_type in MISSING_TYPE_ORDER:
        table = manuscript_performance_table(summary, missing_type)
        if table.empty:
            continue
        per_missing_type[missing_type] = table
        table.to_csv(
            output_dir / f"StudyA_{missing_type}_performance_mean_SD.csv", index=False
        )

    improvement_runs = pd.DataFrame()
    improvement_summary = pd.DataFrame()
    tests = pd.DataFrame()
    table4 = pd.DataFrame()
    if "NoSpatial" in args.configurations and any(
        config in args.configurations for config in ["XY", "Z", "XYZ"]
    ):
        improvement_runs = calculate_improvements(run_metrics)
        improvement_summary = summarize_improvements(improvement_runs)
        tests = paired_tests(run_metrics)
        table4 = manuscript_improvement_table(improvement_summary)
        improvement_runs.to_csv(output_dir / "improvements_by_run.csv", index=False)
        improvement_summary.to_csv(output_dir / "improvements_mean_SD.csv", index=False)
        tests.to_csv(output_dir / "paired_wilcoxon_tests_test_RMSE.csv", index=False)
        table4.to_csv(
            output_dir / "Table4_StudyA_train_test_improvements_mean_SD.csv", index=False
        )

    sheets: dict[str, pd.DataFrame] = {
        "Table2_MT1": table2,
        "Table3_Ablation": table3,
        "Metrics_Summary": summary,
        "Hyperparameters": hyperparameters,
        "Sample_Audit": pd.DataFrame(audit),
    }
    for missing_type, table in per_missing_type.items():
        sheets[f"Performance_{missing_type}"] = table
    if not table4.empty:
        sheets["Table4_Improvements"] = table4
        sheets["Improvement_Numeric"] = improvement_summary
        sheets["Paired_Tests"] = tests
    write_excel(output_dir / "StudyA_manuscript_tables_40_runs.xlsx", sheets)

    pipeline_seconds = time.perf_counter() - pipeline_started
    pd.DataFrame(
        [{
            "stage": "Complete Study A experiment",
            "elapsed_seconds": pipeline_seconds,
            "elapsed_minutes": pipeline_seconds / 60.0,
        }]
    ).to_csv(output_dir / "pipeline_wall_clock_time.csv", index=False)
    print(
        f"Completed in {pipeline_seconds / 60.0:.2f} minutes. "
        f"Results written to {output_dir}",
        flush=True,
    )
    return 0


## Experiment settings

In [9]:
# Edit this cell before running the experiment.
NOTEBOOK_ARGS = [
    "--train", "BM2_Local_BID_11_minus_Table1_XYZ_merged.csv",
    "--validation", "BM2_Local_validation_without_74_matched_records.csv",
    "--test", "BM2_Table_1_md_XYZ_merged_rows.csv",
    "--output-dir", "StudyA_40run_results_Selected22Sept",
    "--runs", "40",
    "--base-seed", "2026",
    "--tuning-seed", "1729",
    "--cv-folds", "5",
    "--cv-strategy", "group",
    "--search-profile", "expanded",
    "--search-iterations", "20",
    "--tuning-score", "rmse",
    "--n-jobs", "-1",
    "--huber-delta", "1.0",
]

# Set to True only when ready for the complete experiment.
# The expanded 56-task analysis performs thousands of model fits.
RUN_FULL_EXPERIMENT = False #False


## Notebook execution helper

In [10]:
def run_pipeline(notebook_args):
    """Run the same argument-driven pipeline safely inside Jupyter."""
    original_argv = sys.argv.copy()
    try:
        sys.argv = ["StudyA_40run_reproducible_tables_mapped_test_rows"] + list(notebook_args)
        return main()
    finally:
        sys.argv = original_argv


## Validate inputs before fitting

This validation also confirms that the test file contains every required `rows` value exactly once.


In [11]:
# Fast data, schema, overlap, and feature-manifest validation.
run_pipeline(NOTEBOOK_ARGS + ["--validate-only"])


Loading and validating Study A data...
                            item            value
             train_rows_supplied             2718
        validation_rows_supplied               36
              test_rows_supplied               20
train_validation_overlapping_ids                0
development_test_overlapping_ids                0
    development_unique_rows_used             2754
           test_unique_rows_used               20
           development_boreholes              236
                  test_boreholes                8
  test_rows_column_unique_values               20
    test_required_rows_available               20
          MT1_test_rows_selected  0, 4, 8, 12, 16
           MT1_test_record_count                5
          MT2_test_rows_selected  1, 5, 9, 13, 17
           MT2_test_record_count                5
          MT3_test_rows_selected 2, 6, 10, 14, 18
           MT3_test_record_count                5
          MT4_test_rows_selected 3, 7, 11, 15, 19
           

0

## Run the complete experiment

The default expanded search directly addresses the reviewer request for broader and equal hyperparameter optimization. For a closer reproduction of the submitted Table 1 grid, change `expanded` to `manuscript`.


In [12]:
if RUN_FULL_EXPERIMENT:
    run_pipeline(NOTEBOOK_ARGS)
else:
    print(
        "Validation is complete. Set RUN_FULL_EXPERIMENT = True in the settings cell "
        "and rerun this cell to launch the complete 40-run analysis."
    )


Validation is complete. Set RUN_FULL_EXPERIMENT = True in the settings cell and rerun this cell to launch the complete 40-run analysis.


## Reproducibility report: system, libraries, and computation time

Run the following cells after the complete experiment. They produce CSV files and a manuscript-ready LaTeX file describing the hardware, software environment, and measured computation times. The timing table distinguishes hyperparameter search, repeated model fitting, diagnostic loss-curve generation, and the complete pipeline wall-clock time.


In [13]:
from importlib import metadata as importlib_metadata
import os
import subprocess

try:
    from IPython.display import display
except ImportError:
    def display(value: Any) -> None:
        print(value)


def notebook_args_namespace(notebook_args: list[str]) -> argparse.Namespace:
    """Parse NOTEBOOK_ARGS without permanently changing the Jupyter kernel arguments."""
    original_argv = sys.argv[:]
    try:
        sys.argv = ["study_a_notebook"] + list(notebook_args)
        return parse_args()
    finally:
        sys.argv = original_argv


def installed_version(distribution: str) -> str:
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


def cpu_model_name() -> str:
    processor = platform.processor().strip()
    if processor:
        return processor
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        for line in cpuinfo.read_text(encoding="utf-8", errors="ignore").splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[-1].strip()
    return "not reported"


def gpu_description() -> str:
    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader",
            ],
            check=True,
            capture_output=True,
            text=True,
            timeout=10,
        )
        values = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
        return "; ".join(values) if values else "not detected"
    except (FileNotFoundError, subprocess.SubprocessError):
        return "not detected"


def system_configuration_table() -> pd.DataFrame:
    try:
        import psutil

        physical_cores = psutil.cpu_count(logical=False)
        logical_cores = psutil.cpu_count(logical=True)
        memory_gib = psutil.virtual_memory().total / (1024 ** 3)
        memory_text = f"{memory_gib:.2f} GiB"
    except ImportError:
        physical_cores = "not reported"
        logical_cores = os.cpu_count() or "not reported"
        memory_text = "not reported (install psutil)"

    rows = [
        ("Operating system", platform.platform()),
        ("Machine architecture", platform.machine() or "not reported"),
        ("CPU model", cpu_model_name()),
        ("Physical CPU cores", physical_cores),
        ("Logical CPU cores", logical_cores),
        ("System memory", memory_text),
        ("GPU name, memory, driver", gpu_description()),
        ("Python implementation", platform.python_implementation()),
        ("Python version", platform.python_version()),
    ]
    return pd.DataFrame(rows, columns=["Component", "Configuration"])


def programming_libraries_table() -> pd.DataFrame:
    distributions = [
        ("NumPy", "numpy"),
        ("pandas", "pandas"),
        ("SciPy", "scipy"),
        ("scikit-learn", "scikit-learn"),
        ("joblib", "joblib"),
        ("XGBoost", "xgboost"),
        ("Matplotlib", "matplotlib"),
        ("SHAP", "shap"),
        ("psutil", "psutil"),
        ("Jupyter", "jupyter"),
        ("IPython kernel", "ipykernel"),
    ]
    return pd.DataFrame(
        [(label, installed_version(distribution)) for label, distribution in distributions],
        columns=["Library", "Version"],
    )


def _timing_summary(
    values: pd.Series,
    stage: str,
    model: str,
    notes: str,
) -> dict[str, Any]:
    numeric = pd.to_numeric(values, errors="coerce").dropna()
    if numeric.empty:
        return {
            "Stage": stage,
            "Model": model,
            "Timed operations": 0,
            "Total seconds": np.nan,
            "Total minutes": np.nan,
            "Mean seconds": np.nan,
            "SD seconds": np.nan,
            "Minimum seconds": np.nan,
            "Maximum seconds": np.nan,
            "Notes": notes,
        }
    return {
        "Stage": stage,
        "Model": model,
        "Timed operations": int(numeric.size),
        "Total seconds": float(numeric.sum()),
        "Total minutes": float(numeric.sum() / 60.0),
        "Mean seconds": float(numeric.mean()),
        "SD seconds": float(numeric.std(ddof=1)) if numeric.size > 1 else 0.0,
        "Minimum seconds": float(numeric.min()),
        "Maximum seconds": float(numeric.max()),
        "Notes": notes,
    }


def computation_time_table(
    output_dir: Path,
    loss_curve_timing: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Aggregate durations without double-counting Development/Test metric rows."""
    records: list[dict[str, Any]] = []

    hyperparameter_path = output_dir / "selected_hyperparameters.csv"
    if hyperparameter_path.is_file():
        tuned = pd.read_csv(hyperparameter_path)
        if "tuning_seconds" in tuned:
            for model, subset in tuned.groupby("model", sort=False):
                records.append(
                    _timing_summary(
                        subset["tuning_seconds"],
                        "Hyperparameter optimization",
                        str(model),
                        "One timed search per target, missing type, and feature configuration.",
                    )
                )

    run_metrics_path = output_dir / "run_metrics_all_models.csv"
    if run_metrics_path.is_file():
        metrics = pd.read_csv(run_metrics_path)
        required = {"split", "model", "fit_seconds"}
        if required.issubset(metrics.columns):
            fits = metrics.loc[
                (metrics["split"] == "Development") & (metrics["model"] != "Ensemble")
            ].copy()
            for model, subset in fits.groupby("model", sort=False):
                records.append(
                    _timing_summary(
                        subset["fit_seconds"],
                        "Repeated model fitting",
                        str(model),
                        "Development rows only; duplicate Test metric rows are excluded.",
                    )
                )

    if loss_curve_timing is not None and not loss_curve_timing.empty:
        for model, subset in loss_curve_timing.groupby("model", sort=False):
            records.append(
                _timing_summary(
                    subset["fit_seconds"],
                    "Multi-run loss-curve diagnostics",
                    str(model),
                    "Borehole-disjoint development/validation split; diagnostic only.",
                )
            )

    wall_clock_path = output_dir / "pipeline_wall_clock_time.csv"
    if wall_clock_path.is_file():
        wall_clock = pd.read_csv(wall_clock_path)
        if {"stage", "elapsed_seconds"}.issubset(wall_clock.columns):
            for _, row in wall_clock.iterrows():
                records.append(
                    _timing_summary(
                        pd.Series([row["elapsed_seconds"]]),
                        str(row["stage"]),
                        "All models",
                        "End-to-end wall-clock time measured with time.perf_counter().",
                    )
                )

    return pd.DataFrame(records)


def latex_escape(value: Any) -> str:
    if pd.isna(value):
        return "--"
    text = f"{value:.3f}" if isinstance(value, (float, np.floating)) else str(value)
    for original, escaped in [
        ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
        ("#", r"\#"), ("_", r"\_"),
    ]:
        text = text.replace(original, escaped)
    return text


def dataframe_to_latex(frame: pd.DataFrame, caption: str, label: str) -> str:
    alignment = "l" + "r" * max(0, len(frame.columns) - 1)
    header = " & ".join(latex_escape(column) for column in frame.columns) + r" \\"
    rows = [
        " & ".join(latex_escape(value) for value in row) + r" \\"
        for row in frame.itertuples(index=False, name=None)
    ]
    return "\n".join([
        r"\begin{table*}[htbp]",
        r"\centering",
        f"\\caption{{{latex_escape(caption)}}}",
        f"\\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        f"\\begin{{tabular}}{{{alignment}}}",
        r"\toprule",
        header,
        r"\midrule",
        *rows,
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\end{table*}",
    ])


def write_reproducibility_report(
    output_dir: Path,
    loss_curve_timing: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    system = system_configuration_table()
    libraries = programming_libraries_table()
    timings = computation_time_table(output_dir, loss_curve_timing)

    system.to_csv(output_dir / "system_configuration.csv", index=False)
    libraries.to_csv(output_dir / "programming_libraries.csv", index=False)
    timings.to_csv(output_dir / "computation_times.csv", index=False)

    latex_parts = [
        r"% Requires \usepackage{booktabs} and \usepackage{graphicx}",
        dataframe_to_latex(
            system,
            "System configuration used for the experiments.",
            "tab:system_configuration",
        ),
        dataframe_to_latex(
            libraries,
            "Principal programming libraries and versions.",
            "tab:programming_libraries",
        ),
        dataframe_to_latex(
            timings,
            "Measured computation times for model development and diagnostics.",
            "tab:computation_times",
        ),
    ]
    (output_dir / "reproducibility_tables.tex").write_text(
        "\n\n".join(latex_parts), encoding="utf-8"
    )
    return system, libraries, timings


## Principal outputs

- `Table2_StudyA_MT1_mean_SD.csv`
- `Table3_StudyA_model_ablation_mean_SD.csv`
- `Table4_StudyA_train_test_improvements_mean_SD.csv`
- `StudyA_manuscript_tables_40_runs.xlsx`
- `summary_metrics_mean_sd_ci95.csv`
- `run_metrics_all_models.csv`
- `test_predictions_all_runs.csv`, containing only the five mapped test rows for each missing type
- `selected_hyperparameters.csv`
- `paired_wilcoxon_tests_test_RMSE.csv`
- `sample_count_and_overlap_audit.csv`, including the row mapping and selected record counts
- `feature_manifest.csv` and `experiment_manifest.json`
- `system_configuration.csv`, `programming_libraries.csv`, and `computation_times.csv`
- `reproducibility_tables.tex`
- `pipeline_wall_clock_time.csv`
- `StudyA_MT1_XYZ_multirun_loss_curves.png` and `.pdf`
- `StudyA_MT1_XYZ_multirun_loss_curve_values.csv` and `_timing.csv`


## Fixed-dataset 40-seed ensemble and spatial-comparator analysis

This section evaluates the XYZ ensemble, local ordinary kriging (OK), and regression--kriging (RK) using the same complete Study A development dataset and the same fixed task-specific held-out test observations in every run.

- The ensemble results are read from the original 40-run experiment. Seeds 2026--2065 control the stochastic fitting of RF, GBR, and XGB.
- For OK, the same seeds control reproducible Monte Carlo sampling of development-coordinate pairs used to estimate the spherical-variogram range.
- For RK, the ridge penalty is selected once from development data and locked. The run seed controls the borehole-grouped out-of-fold partition used to form residuals and the pair sampling used to estimate the residual variogram.

No development or test observations are resampled or removed. The reported standard deviations therefore measure method-specific seed sensitivity on a fixed dataset. Although results are aligned by seed for bookkeeping, the seed controls different stochastic mechanisms across methods, so cross-method differences are reported descriptively rather than as paired inferential tests.


In [14]:
# Imports and fixed settings for the spatial comparators
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict


SPATIAL_COORDINATES = ["x", "y", "z_depth"]
OK_NEIGHBORS = 64
VARIOGRAM_PAIR_SAMPLE = 50_000
VARIOGRAM_RANGE_QUANTILE = 0.75
VARIOGRAM_NUGGET_FRACTION = 0.05
RK_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
SPATIAL_COMPARATOR_RUNS = 40


def _sample_pair_distances(
    coordinates: np.ndarray,
    max_pairs: int,
    seed: int,
) -> np.ndarray:
    """Sample nonzero development-coordinate pair distances reproducibly."""
    coordinates = np.asarray(coordinates, dtype=float)
    count = len(coordinates)
    if count < 2:
        raise ValueError("At least two development observations are required.")
    rng = np.random.default_rng(seed)
    pair_count = min(max_pairs, max(1, count * (count - 1) // 2))
    first = rng.integers(0, count, size=pair_count)
    second = rng.integers(0, count, size=pair_count)
    keep = first != second
    distances = np.linalg.norm(
        coordinates[first[keep]] - coordinates[second[keep]], axis=1
    )
    distances = distances[np.isfinite(distances) & (distances > 0)]
    if distances.size == 0:
        raise ValueError("Development coordinates do not contain nonzero separations.")
    return distances


def estimate_variogram_parameters(
    scaled_coordinates: np.ndarray,
    values: np.ndarray,
    seed: int,
) -> dict[str, float]:
    """Estimate spherical-variogram parameters using development data only."""
    distances = _sample_pair_distances(
        scaled_coordinates, VARIOGRAM_PAIR_SAMPLE, seed
    )
    spatial_range = float(np.quantile(distances, VARIOGRAM_RANGE_QUANTILE))
    variance = float(np.var(np.asarray(values, dtype=float), ddof=1))
    variance = max(variance, np.finfo(float).eps)
    nugget = VARIOGRAM_NUGGET_FRACTION * variance
    partial_sill = max(variance - nugget, np.finfo(float).eps)
    return {
        "range": max(spatial_range, np.finfo(float).eps),
        "partial_sill": partial_sill,
        "nugget": nugget,
    }


def spherical_semivariogram(
    distances: np.ndarray,
    spatial_range: float,
    partial_sill: float,
    nugget: float,
) -> np.ndarray:
    distances = np.asarray(distances, dtype=float)
    ratio = np.clip(distances / spatial_range, 0.0, None)
    inside = nugget + partial_sill * (1.5 * ratio - 0.5 * ratio**3)
    semivariance = np.where(
        distances <= spatial_range,
        inside,
        nugget + partial_sill,
    )
    return np.where(distances == 0.0, 0.0, semivariance)


def local_ordinary_kriging_predict(
    train_coordinates: np.ndarray,
    train_values: np.ndarray,
    query_coordinates: np.ndarray,
    neighbors: int,
    variogram: dict[str, float],
) -> np.ndarray:
    """Predict with local ordinary kriging and an unbiasedness constraint."""
    train_coordinates = np.asarray(train_coordinates, dtype=float)
    train_values = np.asarray(train_values, dtype=float)
    query_coordinates = np.asarray(query_coordinates, dtype=float)
    neighbor_count = min(int(neighbors), len(train_coordinates))
    if neighbor_count < 2:
        raise ValueError("Ordinary kriging requires at least two neighbors.")

    tree = cKDTree(train_coordinates)
    predictions = []
    for query in query_coordinates:
        distances, indices = tree.query(query, k=neighbor_count)
        distances = np.atleast_1d(distances)
        indices = np.atleast_1d(indices)

        exact = np.flatnonzero(distances <= 1e-12)
        if exact.size:
            predictions.append(float(np.mean(train_values[indices[exact]])))
            continue

        local_coordinates = train_coordinates[indices]
        local_values = train_values[indices]
        gamma_matrix = spherical_semivariogram(
            cdist(local_coordinates, local_coordinates),
            variogram["range"],
            variogram["partial_sill"],
            variogram["nugget"],
        )
        gamma_query = spherical_semivariogram(
            cdist(local_coordinates, query.reshape(1, -1)).ravel(),
            variogram["range"],
            variogram["partial_sill"],
            variogram["nugget"],
        )

        system = np.empty((neighbor_count + 1, neighbor_count + 1), dtype=float)
        system[:neighbor_count, :neighbor_count] = gamma_matrix
        system[:neighbor_count, -1] = 1.0
        system[-1, :neighbor_count] = 1.0
        system[-1, -1] = 0.0
        system[:neighbor_count, :neighbor_count] += np.eye(neighbor_count) * 1e-10
        rhs = np.append(gamma_query, 1.0)
        try:
            solution = np.linalg.solve(system, rhs)
        except np.linalg.LinAlgError:
            solution = np.linalg.lstsq(system, rhs, rcond=None)[0]
        predictions.append(float(np.dot(solution[:neighbor_count], local_values)))
    return np.asarray(predictions)


def prepare_spatial_coordinates(
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    development_coordinates = scaler.fit_transform(
        development.loc[:, SPATIAL_COORDINATES]
    )
    evaluation_coordinates = scaler.transform(
        evaluation.loc[:, SPATIAL_COORDINATES]
    )
    return development_coordinates, evaluation_coordinates, scaler


In [15]:
def _rk_feature_columns(task: ExperimentTask) -> list[str]:
    return [
        feature for feature in task.features
        if feature not in SPATIAL_COORDINATES
    ]


def _rk_pipeline(task: ExperimentTask, alpha: float) -> Pipeline:
    return Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=task.polynomial_degree,
                    include_bias=False,
                ),
            ),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def select_locked_rk_alpha(
    task: ExperimentTask,
    development: pd.DataFrame,
    args: argparse.Namespace,
) -> float:
    """Select the RK trend penalty once using development data only."""
    features = _rk_feature_columns(task)
    X = development.loc[:, features]
    y = development.loc[:, task.target].to_numpy(dtype=float)
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development["borehole_id"],
    )
    search = GridSearchCV(
        _rk_pipeline(task, alpha=1.0),
        param_grid={"ridge__alpha": RK_ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=args.n_jobs,
        refit=True,
    )
    fit_kwargs = {"groups": groups} if groups is not None else {}
    search.fit(X, y, **fit_kwargs)
    return float(search.best_params_["ridge__alpha"])


def ordinary_kriging_seeded_predict(
    task: ExperimentTask,
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
    seed: int,
) -> tuple[np.ndarray, dict[str, float]]:
    y = development.loc[:, task.target].to_numpy(dtype=float)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development, evaluation
    )
    variogram = estimate_variogram_parameters(
        development_coordinates, y, seed
    )
    prediction = local_ordinary_kriging_predict(
        development_coordinates,
        y,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=variogram,
    )
    return prediction, variogram


def regression_kriging_seeded_predict(
    task: ExperimentTask,
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
    args: argparse.Namespace,
    seed: int,
    locked_alpha: float,
) -> tuple[np.ndarray, dict[str, float]]:
    """Construct seeded grouped OOF residuals and krige them in XYZ space."""
    features = _rk_feature_columns(task)
    X_development = development.loc[:, features]
    X_evaluation = evaluation.loc[:, features]
    y_development = development.loc[:, task.target].to_numpy(dtype=float)

    trend = _rk_pipeline(task, alpha=locked_alpha)
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        seed,
        development["borehole_id"],
    )
    prediction_kwargs = {
        "estimator": clone(trend),
        "X": X_development,
        "y": y_development,
        "cv": cv,
        "n_jobs": args.n_jobs,
        "method": "predict",
    }
    if groups is not None:
        prediction_kwargs["groups"] = groups
    out_of_fold_trend = cross_val_predict(**prediction_kwargs)
    residuals = y_development - out_of_fold_trend

    trend.fit(X_development, y_development)
    evaluation_trend = trend.predict(X_evaluation)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development, evaluation
    )
    residual_variogram = estimate_variogram_parameters(
        development_coordinates, residuals, seed
    )
    residual_correction = local_ordinary_kriging_predict(
        development_coordinates,
        residuals,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=residual_variogram,
    )
    metadata = {
        "ridge_alpha": locked_alpha,
        "variogram_range": residual_variogram["range"],
        "variogram_partial_sill": residual_variogram["partial_sill"],
        "variogram_nugget": residual_variogram["nugget"],
    }
    return evaluation_trend + residual_correction, metadata


In [16]:
def _simple_latex_table(frame: pd.DataFrame) -> str:
    def clean(value: Any) -> str:
        text = str(value)
        if "$" in text:
            return text
        return text.replace("_", r"\_").replace("%", r"\%")

    lines = [
        r"\begin{tabular}{" + "l" * len(frame.columns) + "}",
        r"\hline",
        " & ".join(clean(column) for column in frame.columns) + r" \\",
        r"\hline",
    ]
    for row in frame.itertuples(index=False, name=None):
        lines.append(" & ".join(clean(value) for value in row) + r" \\")
    lines.extend([r"\hline", r"\end{tabular}", ""])
    return "\n".join(lines)


def run_fixed_dataset_seeded_spatial_comparison(
    notebook_args: list[str],
    runs: int = SPATIAL_COMPARATOR_RUNS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    args = notebook_args_namespace(notebook_args)
    output_dir = Path(args.output_dir).expanduser().resolve()
    comparison_dir = output_dir / "fixed_dataset_40seed_spatial_comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    train = load_and_engineer(resolve_input_path(args.train))
    validation = load_and_engineer(resolve_input_path(args.validation))
    test = load_and_engineer(resolve_input_path(args.test))
    test, _ = validate_test_row_mapping(test)
    _, development, test, split_audit = prepare_splits(
        train, validation, test, args.overlap_policy
    )

    tasks = [
        task for task in iter_tasks(args)
        if task.configuration == "XYZ"
    ]
    if len(tasks) != 14:
        raise ValueError(f"Expected 14 XYZ tasks, found {len(tasks)}.")

    ensemble_path = output_dir / "run_metrics_all_models.csv"
    if not ensemble_path.is_file():
        raise FileNotFoundError(
            f"{ensemble_path} was not found. Run the original complete "
            "40-run ensemble experiment first."
        )
    ensemble_runs = pd.read_csv(ensemble_path)
    ensemble_runs = ensemble_runs.loc[
        (ensemble_runs["configuration"] == "XYZ")
        & (ensemble_runs["model"] == "Ensemble")
        & (ensemble_runs["split"] == "Test")
    ].copy()
    expected_seeds = list(range(args.base_seed, args.base_seed + runs))
    available_seeds = sorted(ensemble_runs["seed"].astype(int).unique().tolist())
    if available_seeds != expected_seeds:
        raise ValueError(
            "The ensemble output does not contain exactly the expected seeds "
            f"{expected_seeds[0]}--{expected_seeds[-1]}."
        )

    locked_alphas = {
        (task.missing_type, task.target): select_locked_rk_alpha(
            task, development, args
        )
        for task in tasks
    }

    metric_rows = []
    prediction_rows = []
    metadata_rows = []
    for task_number, task in enumerate(tasks, start=1):
        task_test = select_test_rows(test, task.missing_type)
        truth = task_test.loc[:, task.target].to_numpy(dtype=float)
        locked_alpha = locked_alphas[(task.missing_type, task.target)]
        for run_index, seed in enumerate(expected_seeds, start=1):
            print(
                f"[{task_number}/{len(tasks)}] {task.missing_type} {task.target}, "
                f"run {run_index}/{runs}, seed={seed}",
                flush=True,
            )
            started = time.perf_counter()
            ok_prediction, ok_variogram = ordinary_kriging_seeded_predict(
                task, development, task_test, seed
            )
            ok_seconds = time.perf_counter() - started

            started = time.perf_counter()
            rk_prediction, rk_metadata = regression_kriging_seeded_predict(
                task,
                development,
                task_test,
                args,
                seed,
                locked_alpha,
            )
            rk_seconds = time.perf_counter() - started

            for method, prediction, seconds, metadata in [
                (
                    "Ordinary Kriging",
                    ok_prediction,
                    ok_seconds,
                    {
                        "ridge_alpha": np.nan,
                        "variogram_range": ok_variogram["range"],
                        "variogram_partial_sill": ok_variogram["partial_sill"],
                        "variogram_nugget": ok_variogram["nugget"],
                    },
                ),
                (
                    "Regression Kriging",
                    rk_prediction,
                    rk_seconds,
                    rk_metadata,
                ),
            ]:
                values = calculate_metrics(truth, prediction, args.huber_delta)
                metric_rows.append(
                    {
                        "run": run_index,
                        "seed": seed,
                        "missing_type": task.missing_type,
                        "target": task.target,
                        "configuration": "XYZ",
                        "method": method,
                        "split": "Test",
                        "n_development": len(development),
                        "n_development_boreholes": development["borehole_id"].nunique(),
                        "n_test": len(task_test),
                        "fit_predict_seconds": seconds,
                        **values,
                    }
                )
                metadata_rows.append(
                    {
                        "run": run_index,
                        "seed": seed,
                        "missing_type": task.missing_type,
                        "target": task.target,
                        "method": method,
                        "neighbors": min(OK_NEIGHBORS, len(development)),
                        **metadata,
                    }
                )
                for position, (_, source_row) in enumerate(task_test.iterrows()):
                    prediction_rows.append(
                        {
                            "run": run_index,
                            "seed": seed,
                            "missing_type": task.missing_type,
                            "target": task.target,
                            "method": method,
                            "test_row": int(source_row["rows"]),
                            "global_id": source_row["global_id"],
                            "borehole_id": source_row["borehole_id"],
                            "observed": float(truth[position]),
                            "predicted": float(prediction[position]),
                            "residual": float(truth[position] - prediction[position]),
                        }
                    )

    comparator_runs = pd.DataFrame(metric_rows)
    predictions = pd.DataFrame(prediction_rows)
    metadata = pd.DataFrame(metadata_rows)

    metric_names = ["rmse", "rmse_percent", "l2", "robust_l2", "mae", "r2"]
    comparator_summary = (
        comparator_runs.groupby(
            ["missing_type", "target", "method"], as_index=False
        )[metric_names]
        .agg(["mean", "std"])
    )
    comparator_summary.columns = [
        "_".join(column).rstrip("_")
        if isinstance(column, tuple) else column
        for column in comparator_summary.columns
    ]
    comparator_summary = comparator_summary.rename(
        columns={"missing_type_": "missing_type", "target_": "target", "method_": "method"}
    )

    ensemble_summary = (
        ensemble_runs.groupby(["missing_type", "target"], as_index=False)[metric_names]
        .agg(["mean", "std"])
    )
    ensemble_summary.columns = [
        "_".join(column).rstrip("_")
        if isinstance(column, tuple) else column
        for column in ensemble_summary.columns
    ]
    ensemble_summary = ensemble_summary.rename(
        columns={"missing_type_": "missing_type", "target_": "target"}
    )

    comparator_rmse = comparator_summary.pivot(
        index=["missing_type", "target"],
        columns="method",
        values=["rmse_mean", "rmse_std"],
    )
    comparator_rmse.columns = [
        f"{method} {stat.replace('rmse_', '').upper()} RMSE"
        for stat, method in comparator_rmse.columns
    ]
    comparator_rmse = comparator_rmse.reset_index()
    comparison = ensemble_summary[
        ["missing_type", "target", "rmse_mean", "rmse_std"]
    ].rename(
        columns={
            "rmse_mean": "Ensemble XYZ MEAN RMSE",
            "rmse_std": "Ensemble XYZ STD RMSE",
        }
    ).merge(
        comparator_rmse,
        on=["missing_type", "target"],
        how="left",
        validate="one_to_one",
    )

    mean_columns = [
        "Ensemble XYZ MEAN RMSE",
        "Ordinary Kriging MEAN RMSE",
        "Regression Kriging MEAN RMSE",
    ]
    method_for_column = {
        "Ensemble XYZ MEAN RMSE": "Ensemble XYZ",
        "Ordinary Kriging MEAN RMSE": "Ordinary Kriging",
        "Regression Kriging MEAN RMSE": "Regression Kriging",
    }
    comparison["Lowest mean RMSE"] = comparison[mean_columns].idxmin(axis=1).map(
        method_for_column
    )

    # Same fixed records are used in every run. Seeds are retained for traceability,
    # but their stochastic meanings differ across the three methods.
    paired_descriptive = []
    ensemble_pairing = ensemble_runs[
        ["seed", "missing_type", "target", "rmse"]
    ].rename(columns={"rmse": "ensemble_rmse"})
    for method in ["Ordinary Kriging", "Regression Kriging"]:
        method_runs = comparator_runs.loc[
            comparator_runs["method"] == method,
            ["seed", "missing_type", "target", "rmse"],
        ].rename(columns={"rmse": "comparator_rmse"})
        joined = ensemble_pairing.merge(
            method_runs,
            on=["seed", "missing_type", "target"],
            how="inner",
            validate="one_to_one",
        )
        joined["comparator"] = method
        joined["ensemble_minus_comparator_rmse"] = (
            joined["ensemble_rmse"] - joined["comparator_rmse"]
        )
        paired_descriptive.append(joined)
    paired_descriptive = pd.concat(paired_descriptive, ignore_index=True)

    audit = pd.DataFrame(
        [
            {
                "runs": runs,
                "seed_start": expected_seeds[0],
                "seed_end": expected_seeds[-1],
                "development_records_each_run": len(development),
                "development_boreholes_each_run": development["borehole_id"].nunique(),
                "test_pool_records_each_run": len(test),
                "test_pool_boreholes_each_run": test["borehole_id"].nunique(),
                "development_resampling": False,
                "test_resampling": False,
                "comparison_basis": "Fixed complete development and test datasets",
            }
        ]
    )

    manuscript_table = comparison[["missing_type", "target"]].copy()
    for method in ["Ensemble XYZ", "Ordinary Kriging", "Regression Kriging"]:
        mean_col = f"{method} MEAN RMSE"
        std_col = f"{method} STD RMSE"
        manuscript_table[method] = comparison.apply(
            lambda row: f'{row[mean_col]:.4f} $\\pm$ {row[std_col]:.4f}',
            axis=1,
        )
    manuscript_table["Lowest mean RMSE"] = comparison["Lowest mean RMSE"]
    manuscript_table = manuscript_table.rename(
        columns={"missing_type": "Missing Type", "target": "Target"}
    )

    comparator_runs.to_csv(comparison_dir / "spatial_comparator_40seed_metrics.csv", index=False)
    predictions.to_csv(comparison_dir / "spatial_comparator_40seed_predictions.csv", index=False)
    metadata.to_csv(comparison_dir / "spatial_comparator_40seed_settings.csv", index=False)
    comparator_summary.to_csv(comparison_dir / "spatial_comparator_summary_mean_sd.csv", index=False)
    ensemble_summary.to_csv(comparison_dir / "ensemble_xyz_summary_mean_sd.csv", index=False)
    comparison.to_csv(comparison_dir / "ensemble_OK_RK_rmse_comparison.csv", index=False)
    paired_descriptive.to_csv(comparison_dir / "seed_aligned_descriptive_differences.csv", index=False)
    audit.to_csv(comparison_dir / "fixed_dataset_audit.csv", index=False)
    pd.DataFrame(
        [
            {"missing_type": key[0], "target": key[1], "locked_ridge_alpha": value}
            for key, value in locked_alphas.items()
        ]
    ).to_csv(comparison_dir / "locked_rk_alphas.csv", index=False)
    (comparison_dir / "ensemble_OK_RK_rmse_comparison.tex").write_text(
        _simple_latex_table(manuscript_table), encoding="utf-8"
    )

    print(f"Outputs written to {comparison_dir}")
    display(audit)
    display(manuscript_table)
    return comparator_runs, comparator_summary, comparison, audit


### Execute the fixed-dataset 40-seed comparison

First run the original complete ensemble experiment so that `run_metrics_all_models.csv` contains the 40 XYZ ensemble runs. Then set the flag below to `True`. For a short validation run, temporarily pass `runs=2`; restore `runs=40` for manuscript results.


In [17]:
RUN_FIXED_DATASET_40SEED_SPATIAL_COMPARISON = True

fixed_seed_comparator_runs = pd.DataFrame()
fixed_seed_comparator_summary = pd.DataFrame()
fixed_seed_rmse_comparison = pd.DataFrame()
fixed_seed_dataset_audit = pd.DataFrame()

if RUN_FIXED_DATASET_40SEED_SPATIAL_COMPARISON:
    (
        fixed_seed_comparator_runs,
        fixed_seed_comparator_summary,
        fixed_seed_rmse_comparison,
        fixed_seed_dataset_audit,
    ) = run_fixed_dataset_seeded_spatial_comparison(
        NOTEBOOK_ARGS,
        runs=SPATIAL_COMPARATOR_RUNS,
    )
else:
    print(
        "Run the original complete ensemble experiment first, then set "
        "RUN_FIXED_DATASET_40SEED_SPATIAL_COMPARISON = True."
    )


[1/14] MT1 Cc, run 1/40, seed=2026
[1/14] MT1 Cc, run 2/40, seed=2027
[1/14] MT1 Cc, run 3/40, seed=2028
[1/14] MT1 Cc, run 4/40, seed=2029
[1/14] MT1 Cc, run 5/40, seed=2030
[1/14] MT1 Cc, run 6/40, seed=2031
[1/14] MT1 Cc, run 7/40, seed=2032
[1/14] MT1 Cc, run 8/40, seed=2033
[1/14] MT1 Cc, run 9/40, seed=2034
[1/14] MT1 Cc, run 10/40, seed=2035
[1/14] MT1 Cc, run 11/40, seed=2036
[1/14] MT1 Cc, run 12/40, seed=2037
[1/14] MT1 Cc, run 13/40, seed=2038
[1/14] MT1 Cc, run 14/40, seed=2039
[1/14] MT1 Cc, run 15/40, seed=2040
[1/14] MT1 Cc, run 16/40, seed=2041
[1/14] MT1 Cc, run 17/40, seed=2042
[1/14] MT1 Cc, run 18/40, seed=2043
[1/14] MT1 Cc, run 19/40, seed=2044
[1/14] MT1 Cc, run 20/40, seed=2045
[1/14] MT1 Cc, run 21/40, seed=2046
[1/14] MT1 Cc, run 22/40, seed=2047
[1/14] MT1 Cc, run 23/40, seed=2048
[1/14] MT1 Cc, run 24/40, seed=2049
[1/14] MT1 Cc, run 25/40, seed=2050
[1/14] MT1 Cc, run 26/40, seed=2051
[1/14] MT1 Cc, run 27/40, seed=2052
[1/14] MT1 Cc, run 28/40, seed=2053
[

,runs,seed_start,seed_end,development_records_each_run,development_boreholes_each_run,test_pool_records_each_run,test_pool_boreholes_each_run,development_resampling,test_resampling,comparison_basis
0,40,2026,2065,2754,236,20,8,False,False,Fixed complete development and test datasets


,Missing Type,Target,Ensemble XYZ,Ordinary Kriging,Regression Kriging,Lowest mean RMSE
0,MT1,Cc,0.1427 $\pm$ 0.0031,0.2383 $\pm$ 0.0000,0.1380 $\pm$ 0.0004,Regression Kriging
1,MT1,Cv,59.7514 $\pm$ 0.7474,64.5316 $\pm$ 0.0041,142.6931 $\pm$ 4.5311,Ensemble XYZ
2,MT1,E50,3338.2252 $\pm$ 32.2306,3086.7326 $\pm$ 0.1803,3017.0659 $\pm$ 13.8410,Regression Kriging
3,MT1,Pc,19.4211 $\pm$ 0.3812,17.5127 $\pm$ 0.0004,37.5228 $\pm$ 1.4827,Ordinary Kriging
4,MT1,su,10.2043 $\pm$ 0.1881,12.0308 $\pm$ 0.0029,14.2170 $\pm$ 0.1840,Ensemble XYZ
5,MT2,Cc,0.0870 $\pm$ 0.0011,0.2245 $\pm$ 0.0000,0.1155 $\pm$ 0.0016,Ensemble XYZ
6,MT2,Cv,34.4001 $\pm$ 1.8756,198.5517 $\pm$ 0.0076,96.4159 $\pm$ 3.7796,Ensemble XYZ
7,MT2,E50,1484.2932 $\pm$ 42.4828,1219.4347 $\pm$ 0.4035,1344.1906 $\pm$ 23.7850,Ordinary Kriging
8,MT2,Pc,17.0830 $\pm$ 0.3797,18.9467 $\pm$ 0.0019,21.1935 $\pm$ 0.7651,Ensemble XYZ
9,MT3,Cc,0.0530 $\pm$ 0.0016,0.1676 $\pm$ 0.0000,0.0724 $\pm$ 0.0012,Ensemble XYZ


### Output interpretation

The output directory is `fixed_dataset_40seed_spatial_comparison/`.

- `fixed_dataset_audit.csv` confirms that development and test observations remain fixed across all runs.
- `spatial_comparator_40seed_metrics.csv` contains run-level OK and RK metrics.
- `spatial_comparator_summary_mean_sd.csv` contains comparator mean and sample SD values.
- `ensemble_xyz_summary_mean_sd.csv` contains the original 40-run ensemble summary.
- `ensemble_OK_RK_rmse_comparison.csv` and `.tex` provide the manuscript comparison.
- `seed_aligned_descriptive_differences.csv` provides seed-aligned differences for descriptive inspection only.

The standard deviations have method-specific interpretations. Ensemble variability arises from stochastic tree fitting. OK variability arises from seeded variogram pair sampling. RK variability arises from seeded borehole-grouped out-of-fold residual construction and variogram pair sampling. The seed-aligned differences should not be treated as paired inferential evidence because the same seed controls different algorithmic mechanisms.
